In [ ]:
# Reproduce nDSM COG -> canopy pipeline (as in app/app_tokyo_cache.py)
import os, sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from geopy import distance
from shapely.geometry import Polygon
import rasterio
from rasterio.enums import Resampling
from rasterio.windows import from_bounds
from pyproj import Transformer, CRS

# Ensure local modules are importable
repo_root = Path.cwd()
if not (repo_root / 'src' / 'voxcity').exists():
    # if notebook cwd is app/, repo root is parent
    repo_root = repo_root.parent
sys.path.append(str(repo_root / 'src'))
sys.path.append(str(repo_root / 'app'))

from voxcity.generator import get_land_cover_grid
from voxcity.utils.visualization import get_land_cover_classes, visualize_numerical_grid_on_map
from tokyo_las import align_ndsm_to_landcover, build_canopy_from_ndsm, remove_local_spikes_in_canopy

def ndsm_average_grid_from_geotiff(tiff_path: str, meshsize: float, rectangle_vertices):
    # Equivalent to _ndsm_average_grid_from_geotiff in app/app_tokyo_cache.py
    with rasterio.open(tiff_path) as src:
        if src.crs is None:
            raise ValueError('Raster has no CRS')
        to_src = Transformer.from_crs('EPSG:4326', src.crs, always_xy=True)
        minx = min(v[0] for v in rectangle_vertices); maxx = max(v[0] for v in rectangle_vertices)
        miny = min(v[1] for v in rectangle_vertices); maxy = max(v[1] for v in rectangle_vertices)
        (minx, miny) = to_src.transform(float(minx), float(miny))
        (maxx, maxy) = to_src.transform(float(maxx), float(maxy))
        minx, maxx = (min(minx, maxx), max(minx, maxx))
        miny, maxy = (min(miny, maxy), max(miny, maxy))
        win = from_bounds(minx, miny, maxx, maxy, src.transform)
        win = win.round_offsets().round_lengths()
        width_m = maxx - minx
        height_m = maxy - miny
        out_w = max(1, int(round(width_m / float(meshsize))))
        out_h = max(1, int(round(height_m / float(meshsize))))
        arr = src.read(1, window=win, out_shape=(out_h, out_w), resampling=Resampling.average)
        nodata = src.nodata
        arr = arr.astype(float, copy=False)
        if nodata is not None:
            arr = np.where(arr == float(nodata), np.nan, arr)
        return arr

def rectangle_from_center(center_lon, center_lat, width_m, height_m):
    north = distance.distance(meters=height_m / 2.0).destination((center_lat, center_lon), bearing=0)
    south = distance.distance(meters=height_m / 2.0).destination((center_lat, center_lon), bearing=180)
    east = distance.distance(meters=width_m / 2.0).destination((center_lat, center_lon), bearing=90)
    west = distance.distance(meters=width_m / 2.0).destination((center_lat, center_lon), bearing=270)
    return [
        (west.longitude, south.latitude),
        (west.longitude, north.latitude),
        (east.longitude, north.latitude),
        (east.longitude, south.latitude),
    ]


In [ ]:
# Define AOI around Imperial Palace (near Tokyo Station)
# Center roughly: 35.685175, 139.752799 (north palace garden area)
center_lat = 35.685175
center_lon = 139.752799
width_m = 1500  # adjust as needed
height_m = 1500
meshsize = 5

rectangle_vertices = rectangle_from_center(center_lon, center_lat, width_m, height_m)
rectangle_vertices


> **Archived, and the `tree_id` chain below is superseded and unsafe — do not copy it.**
>
> `name_to_id.get('Tree') or name_to_id.get('Trees') or ... or 4` treats index **0** as missing.
> ESA WorldCover has `Trees` at index **0**, so that source falls through the whole chain to the
> hard-coded `4` — which in ESA WorldCover is **`Built-up`**, writing canopy heights onto buildings.
>
> It is harmless *here* only because this notebook is OpenEarthMapJapan-only, where `Tree`
> happens to sit at index 4 as well (see the cell near the end that spells out
> `tree_id = 4, building_id = 7`). The same chain appears again further down this notebook.
>
> The corrected resolver — explicit `is not None`, no numeric fallback into a foreign class —
> lives in `app/backend/main.py` as `_resolve_tree_id`.


In [ ]:
# Compute land cover grid using the same source logic as the app
# For Japan, the app uses 'OpenEarthMapJapan' automatically
output_dir = str((repo_root / 'app' / 'data' / 'temp').resolve())
os.makedirs(output_dir, exist_ok=True)
land_cover_source = 'OpenEarthMapJapan'

land_cover_grid = get_land_cover_grid(
    rectangle_vertices,
    meshsize,
    land_cover_source,
    output_dir,
    gridvis=False,
)

classes = get_land_cover_classes(land_cover_source)
name_to_id = {name: i for i, name in enumerate(classes.values())}
tree_id = (
    name_to_id.get('Tree')
    or name_to_id.get('Trees')
    or name_to_id.get('Tree Canopy')
    or 4
)
if 'Building' in name_to_id:
    building_id = name_to_id['Building']
elif 'Buildings' in name_to_id:
    building_id = name_to_id['Buildings']
elif 'Building Footprint' in name_to_id:
    building_id = name_to_id['Building Footprint']
else:
    building_id = 1

land_cover_grid.shape, tree_id, building_id


In [ ]:
# Load nDSM from COG/TIFF and average to mesh
ndsm_cog = str((repo_root / 'app' / 'data' / 'temp' / 'ndsm_cog.tif').resolve())
ndsm_cached = str((repo_root / 'app' / 'data' / 'temp' / 'ndsm.tif').resolve())

if os.path.exists(ndsm_cog):
    ndsm_grid = ndsm_average_grid_from_geotiff(ndsm_cog, meshsize, rectangle_vertices)
elif os.path.exists(ndsm_cached):
    ndsm_grid = ndsm_average_grid_from_geotiff(ndsm_cached, meshsize, rectangle_vertices)
else:
    raise FileNotFoundError('nDSM COG/TIFF not found in app/data/temp')

ndsm_grid.shape, np.nanmin(ndsm_grid), np.nanmax(ndsm_grid), np.isnan(ndsm_grid).mean()


In [ ]:
# Align nDSM to land cover grid and build canopy
ndsm_aligned, align_info = align_ndsm_to_landcover(
    ndsm_grid,
    land_cover_grid,
    tree_value=tree_id,
    allow_resample=True,
    try_vertical_flip=True,
)
print('align_info:', align_info)

# Initial canopy: keep nDSM at tree cells; NaN elsewhere
canopy_initial = build_canopy_from_ndsm(
    ndsm_aligned,
    land_cover_grid,
    tree_value=tree_id,
    non_tree_fill=np.nan,
    clamp_negative_to_zero=True,
)

# Fill missing tree cells with static height ONLY (same as app_tokyo_cache.py refinement)
static_tree_height = 10.0
canopy_refined = canopy_initial.copy().astype(float)
tree_mask_local = (land_cover_grid == tree_id)
missing_tree = tree_mask_local & np.isnan(canopy_refined)
if np.any(missing_tree):
    canopy_refined[missing_tree] = float(static_tree_height)

# Remove local spikes near buildings
canopy_refined = remove_local_spikes_in_canopy(
    canopy_refined,
    land_cover_grid,
    tree_value=tree_id,
    building_value=building_id,
    high_threshold_m=12.0,
    building_buffer_m=20.0,
    cell_size_m=float(meshsize),
    replacement_tree_height_m=None,
)

np.nanmin(canopy_refined), np.nanmax(canopy_refined), np.isnan(canopy_refined).mean()


In [ ]:
# Diagnostics: where nDSM is NaN vs where canopy got static height
ndsm_nan = np.isnan(ndsm_aligned)
filled_static = (land_cover_grid == tree_id) & np.isnan(canopy_initial)

# Color limits for visualization
vmin_val = 0.0
vmax_val = 20.0  # adjust as needed

print('ndsm_nan fraction:', ndsm_nan.mean())
print('filled_static fraction (over all cells):', filled_static.mean())
print('filled_static over tree cells:', filled_static[land_cover_grid == tree_id].mean() if np.any(land_cover_grid == tree_id) else 'n/a')

# Quick masked previews
plt.figure(figsize=(12,3))
plt.subplot(1,3,1); plt.title('nDSM aligned (NaN white)')
plt.imshow(np.where(ndsm_aligned>0, ndsm_aligned, np.nan), cmap='viridis'); plt.colorbar(fraction=0.046)
plt.subplot(1,3,2); plt.title('Canopy initial (NaN white)')
plt.imshow(canopy_initial, cmap='Greens', vmin=vmin_val, vmax=vmax_val); plt.colorbar(fraction=0.046)
plt.subplot(1,3,3); plt.title('Canopy refined (NaN white)')
plt.imshow(canopy_refined, cmap='Greens', vmin=vmin_val, vmax=vmax_val); plt.colorbar(fraction=0.046)
plt.tight_layout()
plt.show()


In [ ]:
# Optional: visualize canopy on basemap like the app overlays ground grids
try:
    visualize_numerical_grid_on_map(
        canopy_refined,
        rectangle_vertices,
        meshsize,
        type='Canopy (m)',
        vmin=vmin_val,
        vmax=vmax_val,
        alpha=0.6,
        edge=False,
        basemap='CartoDB light',
    )
except Exception as e:
    print('Basemap visualization skipped:', e)


In [ ]:
from pathlib import Path
import os
import numpy as np
import rasterio
from pyproj import Transformer

# Prefer the cache path used by precompute_las_cache.py (relative to this notebook in app/)
DEFAULT_PATHS = [
    Path("data/temp/ndsm_cog.tif"),
    Path("data/temp/ndsm.tif"),
]

# Allow overriding via env var
env_path = os.environ.get("NDSM_COG_PATH")
if env_path:
    tif_path = Path(env_path)
else:
    tif_path = next((p for p in DEFAULT_PATHS if p.exists()), Path("data/temp/ndsm_cog.tif"))

print(f"Using: {tif_path}")


In [ ]:
# AOI around Tokyo Station and raster window
center_lonlat = (139.75, 35.681236)
width_m = 2000
height_m = 2000

with rasterio.open(str(tif_path)) as src:
    transformer = Transformer.from_crs(4326, src.crs, always_xy=True)
    cx, cy = transformer.transform(center_lonlat[0], center_lonlat[1])
    half_w = width_m / 2.0
    half_h = height_m / 2.0
    left, right = cx - half_w, cx + half_w
    bottom, top = cy - half_h, cy + half_h

    # Clamp to dataset bounds
    b = src.bounds
    left, right = max(left, b.left), min(right, b.right)
    bottom, top = max(bottom, b.bottom), min(top, b.top)

    from rasterio.windows import from_bounds
    aoi_window = from_bounds(left, bottom, right, top, transform=src.transform).round_offsets().round_lengths()

print("Window (row_off,col_off,height,width):", int(aoi_window.row_off), int(aoi_window.col_off), int(aoi_window.height), int(aoi_window.width))


In [ ]:
# Reproject AOI to WGS84 and normalize (NaN-preserving)
import affine
from rasterio.warp import reproject, Resampling

vmin, vmax = 0.0, 30.0

with rasterio.open(str(tif_path)) as src:
    src_data = src.read(1, window=aoi_window)
    src_transform = rasterio.windows.transform(aoi_window, src.transform)
    src_crs = src.crs
    src_nodata = src.nodata

# Prepare destination
from pyproj import Transformer as _T
_t = _T.from_crs(src_crs, "EPSG:4326", always_xy=True)
left, bottom = _t.transform(src_transform.c, src_transform.f + src_transform.e * src_data.shape[0])
right, top = _t.transform(src_transform.c + src_transform.a * src_data.shape[1], src_transform.f)
dst_width, dst_height = src_data.shape[1], src_data.shape[0]
dst_transform = affine.Affine((right - left) / dst_width, 0, left, 0, -(top - bottom) / dst_height, top)

dst = np.full((dst_height, dst_width), np.nan, dtype="float32")
reproject(
    source=src_data,
    destination=dst,
    src_transform=src_transform,
    src_crs=src_crs,
    src_nodata=src_nodata,
    dst_transform=dst_transform,
    dst_crs="EPSG:4326",
    dst_nodata=np.nan,
    resampling=Resampling.bilinear,
)

# Normalize with clipping; keep NaNs
valid = np.isfinite(dst)
clipped = np.clip(dst, vmin, vmax, where=valid, out=np.full_like(dst, np.nan))
dst_norm = np.where(valid, (clipped - vmin) / max(vmax - vmin, 1e-6), np.nan)

# Leaflet bounds
height, width = dst.shape
west = dst_transform.c
north = dst_transform.f
east = west + dst_transform.a * width
south = north + dst_transform.e * height
leaflet_bounds = [[south, west], [north, east]]

print("Bounds:", leaflet_bounds)


In [ ]:
# Folium overlay with NaNs transparent (viridis)
import folium
from folium.raster_layers import ImageOverlay
import matplotlib.cm as cm

viridis = cm.get_cmap('viridis')
rgba_float = viridis(np.nan_to_num(dst_norm, nan=-1.0))  # map NaN to out-of-range; we'll fix alpha
rgba = (rgba_float * 255).astype(np.uint8)
# Alpha: only finite pixels opaque
alpha = np.where(np.isfinite(dst_norm), 255, 0).astype(np.uint8)
rgba[..., 3] = alpha

center_lat = (leaflet_bounds[0][0] + leaflet_bounds[1][0]) / 2
center_lon = (leaflet_bounds[0][1] + leaflet_bounds[1][1]) / 2
m = folium.Map(location=[center_lat, center_lon], zoom_start=15, tiles='CartoDB positron')

ImageOverlay(
    image=rgba,
    bounds=leaflet_bounds,
    opacity=1.0,
    name='nDSM viridis (0-30 m)',
    cross_origin=False,
).add_to(m)

folium.LayerControl().add_to(m)

m


In [ ]:
from pathlib import Path
import os
import warnings

import numpy as np
import rasterio
import matplotlib.pyplot as plt
import matplotlib

# Prefer the cache path used by precompute_las_cache.py
DEFAULT_PATHS = [
    Path("data/temp/ndsm_cog.tif"),
    Path("data/temp/ndsm.tif"),
]

# Allow overriding via env var
env_path = os.environ.get("NDSM_COG_PATH")
if env_path:
    tif_path = Path(env_path)
else:
    tif_path = next((p for p in DEFAULT_PATHS if p.exists()), Path("data/temp/ndsm_cog.tif"))

print(f"Using: {tif_path}")


In [ ]:
# Define AOI around Tokyo Station (WGS84) and compute raster window
from pyproj import Transformer

# Tokyo Station approx coordinates
center_lonlat = (139.755, 35.682)
# AOI size in meters
width_m = 2000  # east-west
height_m = 2000  # north-south

with rasterio.open(str(tif_path)) as src:
    # Transform center to dataset CRS (meters in EPSG:6677)
    dst_crs = src.crs
    transformer = Transformer.from_crs(4326, dst_crs, always_xy=True)
    cx, cy = transformer.transform(center_lonlat[0], center_lonlat[1])

    half_w = width_m / 2.0
    half_h = height_m / 2.0
    left = cx - half_w
    right = cx + half_w
    bottom = cy - half_h
    top = cy + half_h

    # Clamp to dataset bounds
    bounds = src.bounds
    left = max(left, bounds.left)
    right = min(right, bounds.right)
    bottom = max(bottom, bounds.bottom)
    top = min(top, bounds.top)

    aoi_bounds = (left, bottom, right, top)
    aoi_window = rasterio.windows.from_bounds(left, bottom, right, top, transform=src.transform)
    aoi_window = aoi_window.round_offsets().round_lengths()

    # Report
    print("AOI bounds (dst CRS):", aoi_bounds)
    print("Window offsets (row_off, col_off):", int(aoi_window.row_off), int(aoi_window.col_off))
    print("Window size (height, width):", int(aoi_window.height), int(aoi_window.width))


In [ ]:
# Reproject AOI window to WGS84 and normalize with vmax
import affine
from rasterio.warp import reproject, Resampling, calculate_default_transform
from rasterio.transform import array_bounds

# Parameters for visualization
vmin = 0.0
vmax = 30.0  # cap at 30 m per request
output_res_deg = None  # None keeps approx native res in degrees

with rasterio.open(str(tif_path)) as src:
    # Read AOI window
    data = src.read(1, window=aoi_window)
    src_transform = rasterio.windows.transform(aoi_window, src.transform)
    src_crs = src.crs

    # Target CRS
    dst_crs = "EPSG:4326"

    # Compute target transform and shape
    if output_res_deg is None:
        # approximate resolution in degrees based on window bounds and size
        left, bottom, right, top = rasterio.transform.array_bounds(data.shape[0], data.shape[1], src_transform)
        # transform bounds to WGS84
        from pyproj import Transformer
        transformer = Transformer.from_crs(src_crs, dst_crs, always_xy=True)
        lons = []
        lats = []
        for x, y in [(left, bottom), (left, top), (right, bottom), (right, top)]:
            lon, lat = transformer.transform(x, y)
            lons.append(lon); lats.append(lat)
        min_lon, max_lon = min(lons), max(lons)
        min_lat, max_lat = min(lats), max(lats)
        # keep pixel count similar to source for decent clarity
        dst_width = data.shape[1]
        dst_height = data.shape[0]
        dst_transform = affine.Affine((max_lon - min_lon) / dst_width, 0, min_lon, 0, -(max_lat - min_lat) / dst_height, max_lat)
    else:
        dst_transform, dst_width, dst_height = calculate_default_transform(src_crs, dst_crs, data.shape[1], data.shape[0], *rasterio.transform.array_bounds(data.shape[0], data.shape[1], src_transform), resolution=output_res_deg)

    dst = np.zeros((dst_height, dst_width), dtype="float32")
    reproject(
        source=data,
        destination=dst,
        src_transform=src_transform,
        src_crs=src_crs,
        dst_transform=dst_transform,
        dst_crs=dst_crs,
        resampling=Resampling.bilinear,
        num_threads=2,
    )

# Mask nodata/invalid and normalize
valid = np.isfinite(dst)
dst_norm = np.zeros_like(dst, dtype="float32")
if vmin is not None or vmax is not None:
    lo = vmin if vmin is not None else float(np.nanmin(dst[valid]))
    hi = vmax if vmax is not None else float(np.nanmax(dst[valid]))
else:
    lo, hi = float(np.nanmin(dst[valid])), float(np.nanmax(dst[valid]))

# clip then scale to 0-1
clipped = np.clip(dst, lo, hi)
if hi > lo:
    dst_norm = (clipped - lo) / (hi - lo)
else:
    dst_norm[:] = 0.0

# Build bounds for folium (lat-lon)
height, width = dst.shape
west = dst_transform.c
north = dst_transform.f
east = west + dst_transform.a * width
south = north + dst_transform.e * height
leaflet_bounds = [[south, west], [north, east]]

print("Normalized to [0,1] with vmin=", lo, "vmax=", hi)
print("Leaflet bounds:", leaflet_bounds)


In [ ]:
# Set vmin/vmax and recompute normalization for viridis overlay
import numpy as np
import matplotlib

vmin = 0.0
vmax = 30.0

valid = np.isfinite(dst)
lo, hi = float(vmin), float(vmax)
clipped = np.clip(dst, lo, hi)
if hi > lo:
    dst_norm = (clipped - lo) / (hi - lo)
else:
    dst_norm = np.zeros_like(dst, dtype="float32")

print("Recomputed normalization with vmin=", lo, "vmax=", hi)


In [ ]:
# Folium overlay using viridis colormap
import folium
from folium.raster_layers import ImageOverlay
import matplotlib.cm as cm

# Map normalized [0,1] to viridis RGBA
viridis = cm.get_cmap('viridis')
rgba_float = viridis(np.clip(dst_norm, 0, 1))  # shape (H,W,4), floats 0-1
rgba = (rgba_float * 255).astype(np.uint8)

# Make nodata transparent
alpha = np.where(np.isfinite(dst), 255, 0).astype(np.uint8)
rgba[..., 3] = alpha

center_lat = (leaflet_bounds[0][0] + leaflet_bounds[1][0]) / 2
center_lon = (leaflet_bounds[0][1] + leaflet_bounds[1][1]) / 2
m = folium.Map(location=[center_lat, center_lon], zoom_start=15, tiles='CartoDB positron')

ImageOverlay(
    image=rgba,
    bounds=leaflet_bounds,
    opacity=0.85,
    name='nDSM viridis (0-30 m)',
    cross_origin=False,
).add_to(m)

folium.LayerControl().add_to(m)

m


In [ ]:
# Reproduce canopy pipeline: parameters and utilities
import sys
from pathlib import Path
import os
import numpy as np
import rasterio
from rasterio.windows import from_bounds
from rasterio.enums import Resampling
from pyproj import Transformer, CRS

# Ensure project modules are importable
project_root = Path.cwd().parents[0]  # repo root
sys.path.insert(0, str(project_root / 'src'))  # for voxcity
sys.path.insert(0, str(Path.cwd()))            # for tokyo_las.py in app/

# Imports from the project
try:
    from voxcity.utils.visualization import get_land_cover_classes
except Exception as e:
    raise RuntimeError(f"Failed to import voxcity: {e}")

# Optional utilities from tokyo_las; fallback to no-op/simple behavior if unavailable
try:
    from tokyo_las import remove_local_spikes_in_canopy, align_ndsm_to_landcover, build_canopy_from_ndsm
except Exception:
    def remove_local_spikes_in_canopy(canopy, land_cover, tree_value, building_value, high_threshold_m=12.0, building_buffer_m=20.0, cell_size_m=5.0, replacement_tree_height_m=None):
        return canopy
    def align_ndsm_to_landcover(ndsm_grid, land_cover_grid, tree_value=None, allow_resample=True, try_vertical_flip=True):
        return ndsm_grid, {"aligned": False}
    def build_canopy_from_ndsm(ndsm_aligned, land_cover_grid, tree_value, non_tree_fill=np.nan, clamp_negative_to_zero=True):
        canopy = ndsm_aligned.copy().astype(float)
        canopy[land_cover_grid != int(tree_value)] = non_tree_fill
        if clamp_negative_to_zero:
            neg_mask = (land_cover_grid == int(tree_value)) & np.isfinite(canopy) & (canopy < 0.0)
            canopy[neg_mask] = 0.0
        return canopy

# Files
ndsm_tif = Path("data/temp/ndsm_cog.tif") if Path("data/temp/ndsm_cog.tif").exists() else Path("data/temp/ndsm.tif")
landcover_tif = Path("data/temp/land_cover.tif")
if not ndsm_tif.exists():
    raise FileNotFoundError(f"nDSM TIFF not found: {ndsm_tif}")
if not landcover_tif.exists():
    raise FileNotFoundError(f"Land cover TIFF not found: {landcover_tif}")

# Parameters (match app defaults)
meshsize = 5.0  # meters per cell
static_tree_height = 10.0  # m for missing tree nDSM
high_threshold_m = 12.0
building_buffer_m = 20.0

# Use the geographic AOI already defined above (leaflet_bounds)
# rectangle_vertices = [(lon_min, lat_min), (lon_min, lat_max), (lon_max, lat_max), (lon_max, lat_min)]
lon_min, lat_min = float(min(leaflet_bounds[0][1], leaflet_bounds[1][1])), float(min(leaflet_bounds[0][0], leaflet_bounds[1][0]))
lon_max, lat_max = float(max(leaflet_bounds[0][1], leaflet_bounds[1][1])), float(max(leaflet_bounds[0][0], leaflet_bounds[1][0]))
rectangle_vertices = [(lon_min, lat_min), (lon_min, lat_max), (lon_max, lat_max), (lon_max, lat_min)]

# Resolve land cover class ids (OpenEarthMapJapan). Mirror app logic by enumerating class names
land_cover_source = 'OpenEarthMapJapan'
lc_classes = get_land_cover_classes(land_cover_source)
name_to_id = {name: i for i, name in enumerate(lc_classes.values())}
tree_id = name_to_id.get('Tree') or name_to_id.get('Trees') or name_to_id.get('Tree Canopy') or 4
if 'Building' in name_to_id:
    building_id = name_to_id['Building']
elif 'Buildings' in name_to_id:
    building_id = name_to_id['Buildings']
elif 'Building Footprint' in name_to_id:
    building_id = name_to_id['Building Footprint']
else:
    building_id = 1

print(f"Configured: tree_id={tree_id}, building_id={building_id}, meshsize={meshsize} m")


In [ ]:
# Download OpenEarthMap Japan (OEMJ) GeoTIFF for current AOI
from voxcity.downloader.oemj import save_oemj_as_geotiff
from pathlib import Path
import os

# Ensure output directory exists
landcover_tif.parent.mkdir(parents=True, exist_ok=True)

# Download OEMJ into landcover_tif for our rectangle_vertices
# Fallback: retry with relaxed SSL if strict verify fails
try:
    save_oemj_as_geotiff(rectangle_vertices, str(landcover_tif), zoom=16)
except Exception as e:
    print(f"Strict OEMJ download failed: {e}. Retrying with relaxed SSL...")
    try:
        save_oemj_as_geotiff(
            rectangle_vertices, str(landcover_tif), zoom=16,
            ssl_verify=False, allow_insecure_ssl=True, allow_http_fallback=True, timeout_s=45
        )
    except Exception as e2:
        raise RuntimeError(f"OEMJ download failed: {e2}")

print(f"OEMJ GeoTIFF saved to: {landcover_tif}")


In [ ]:
# Compute canopy grid from nDSM aligned to land cover
import numpy as np
import rasterio
from rasterio.enums import Resampling
from shapely.geometry import Polygon

# 1) Read and average nDSM within AOI at meshsize resolution (projected CRS)
with rasterio.open(str(ndsm_tif)) as src:
    if src.crs is None:
        raise ValueError("nDSM raster has no CRS")
    to_src = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
    minx, miny = to_src.transform(lon_min, lat_min)
    maxx, maxy = to_src.transform(lon_max, lat_max)
    # ensure min<max
    minx, maxx = (min(minx, maxx), max(minx, maxx))
    miny, maxy = (min(miny, maxy), max(miny, maxy))

    win = from_bounds(minx, miny, maxx, maxy, src.transform).round_offsets().round_lengths()

    width_m = maxx - minx
    height_m = maxy - miny
    out_w = max(1, int(round(width_m / float(meshsize))))
    out_h = max(1, int(round(height_m / float(meshsize))))

    ndsm_grid = src.read(1, window=win, out_shape=(out_h, out_w), resampling=Resampling.average).astype(float)
    nodata = src.nodata
    if nodata is not None:
        ndsm_grid = np.where(ndsm_grid == float(nodata), np.nan, ndsm_grid)

# 2) Read land cover subset in the same AOI and resample to ndsm_grid shape
with rasterio.open(str(landcover_tif)) as lc_src:
    if lc_src.crs is None:
        raise ValueError("Land cover raster has no CRS")
    to_lc = Transformer.from_crs("EPSG:4326", lc_src.crs, always_xy=True)
    lminx, lminy = to_lc.transform(lon_min, lat_min)
    lmaxx, lmaxy = to_lc.transform(lon_max, lat_max)
    lminx, lmaxx = (min(lminx, lmaxx), max(lminx, lmaxx))
    lminy, lmaxy = (min(lminy, lmaxy), max(lminy, lmaxy))

    lwin = from_bounds(lminx, lminy, lmaxx, lmaxy, lc_src.transform).round_offsets().round_lengths()

    if lc_src.count >= 3:
        # RGB classification GeoTIFF: read 3 bands and classify to OEMJ indices via nearest color
        r = lc_src.read(1, window=lwin, out_shape=ndsm_grid.shape, resampling=Resampling.nearest)
        g = lc_src.read(2, window=lwin, out_shape=ndsm_grid.shape, resampling=Resampling.nearest)
        b = lc_src.read(3, window=lwin, out_shape=ndsm_grid.shape, resampling=Resampling.nearest)
        rgb = np.stack([r, g, b], axis=-1).astype(np.int16)
        from voxcity.utils.lc import get_land_cover_classes as _glc
        oemj_classes = _glc('OpenEarthMapJapan')
        colors = np.array(list(oemj_classes.keys()), dtype=np.int16)   # shape (K,3)
        # Vectorized nearest color
        diff = rgb[..., None, :] - colors[None, None, :, :]            # (H,W,K,3)
        dist2 = np.sum(diff * diff, axis=-1)                           # (H,W,K)
        land_cover_grid = np.argmin(dist2, axis=-1).astype(int)        # OEMJ indices 0..K-1
    else:
        # Single-band indexed raster: assume values are OEMJ indices 0..7
        lc_arr = lc_src.read(1, window=lwin, out_shape=ndsm_grid.shape, resampling=Resampling.nearest)
        land_cover_grid = lc_arr.astype(int, copy=False)

# Debug: report class distribution and tree cell count
uniq, cnt = np.unique(land_cover_grid, return_counts=True)
print({int(u): int(c) for u, c in zip(uniq, cnt)})
print("tree cells:", int((land_cover_grid == int(tree_id)).sum()))

# 3) Build initial canopy from nDSM: tree cells keep nDSM (negative clamped to 0), others NaN
ndsm_aligned, _align_info = align_ndsm_to_landcover(ndsm_grid, land_cover_grid, tree_value=int(tree_id), allow_resample=True, try_vertical_flip=True)
canopy_initial = build_canopy_from_ndsm(
    ndsm_aligned,
    land_cover_grid,
    tree_value=int(tree_id),
    non_tree_fill=np.nan,
    clamp_negative_to_zero=True,
)

# 4) Fill missing tree cells (NaN in tree locations) with static height
canopy_refined = canopy_initial.copy().astype(float)
missing_tree = (land_cover_grid == int(tree_id)) & np.isnan(canopy_refined)
if np.any(missing_tree):
    canopy_refined[missing_tree] = float(static_tree_height)

# 5) Remove local spikes near buildings (if utility available)
canopy_refined = remove_local_spikes_in_canopy(
    canopy_refined,
    land_cover_grid,
    tree_value=int(tree_id),
    building_value=int(building_id),
    high_threshold_m=float(high_threshold_m),
    building_buffer_m=float(building_buffer_m),
    cell_size_m=float(meshsize),
    replacement_tree_height_m=None,
)

# 6) Replace remaining NaN with 0 for downstream processing
canopy_refined = np.nan_to_num(canopy_refined, nan=0.0)

# 7) Compute canopy bottom as fixed ratio (match app logic)
trunk_height_ratio = 11.76 / 19.98
canopy_bottom = np.minimum(canopy_refined * float(trunk_height_ratio), canopy_refined)

print("canopy_refined stats: min=", np.nanmin(canopy_refined), "max=", np.nanmax(canopy_refined))


In [ ]:
from voxcity.utils.lc import get_land_cover_classes as _glc
_oemj = _glc('OpenEarthMapJapan')
_oemj_vals = list(_oemj.values())  # ['Bareland','Rangeland','Developed space','Road','Tree','Water','Agriculture land','Building']

tree_id = _oemj_vals.index('Tree')         # 4
building_id = _oemj_vals.index('Building') # 7
print("IDs:", {"tree_id": tree_id, "building_id": building_id})

# sanity check
print("class counts:", dict(zip(*np.unique(land_cover_grid, return_counts=True))))
print("tree cells:", int((land_cover_grid == tree_id).sum()))

In [ ]:
# Visualize OEMJ land cover and tree mask on Folium
import folium
from folium.raster_layers import ImageOverlay
import numpy as np

# 1) Render OEMJ land cover RGB (from downloaded GeoTIFF)
with rasterio.open(str(landcover_tif)) as lc_src:
    to_lc = Transformer.from_crs("EPSG:4326", lc_src.crs, always_xy=True)
    lminx, lminy = to_lc.transform(lon_min, lat_min)
    lmaxx, lmaxy = to_lc.transform(lon_max, lat_max)
    lminx, lmaxx = (min(lminx, lmaxx), max(lminx, lmaxx))
    lminy, lmaxy = (min(lminy, lmaxy), max(lminy, lmaxy))
    lwin = from_bounds(lminx, lminy, lmaxx, lmaxy, lc_src.transform).round_offsets().round_lengths()

    if lc_src.count >= 3:
        r = lc_src.read(1, window=lwin, out_shape=land_cover_grid.shape, resampling=Resampling.nearest)
        g = lc_src.read(2, window=lwin, out_shape=land_cover_grid.shape, resampling=Resampling.nearest)
        b = lc_src.read(3, window=lwin, out_shape=land_cover_grid.shape, resampling=Resampling.nearest)
        rgb = np.stack([r, g, b], axis=-1).astype(np.uint8)
    else:
        # If single band, colorize by class with simple palette
        pal = {
            0: (128, 0, 0), 1: (0, 255, 36), 2: (148, 148, 148), 3: (255, 255, 255),
            4: (34, 97, 38), 5: (0, 69, 255), 6: (75, 181, 73), 7: (222, 31, 7)
        }
        rgb = np.zeros((*land_cover_grid.shape, 3), dtype=np.uint8)
        for cls, col in pal.items():
            rgb[land_cover_grid == cls] = np.array(col, dtype=np.uint8)

# 2) Build tree-only binary mask from land_cover_grid (tree_id == 4)
tree_mask = (land_cover_grid == int(tree_id)).astype(np.uint8)

# 3) Create overlays
bounds = [[lat_min, lon_min], [lat_max, lon_max]]
center_lat = (lat_min + lat_max) / 2.0
center_lon = (lon_min + lon_max) / 2.0

m_oemj = folium.Map(location=[center_lat, center_lon], zoom_start=16, tiles='CartoDB positron')

# Land cover layer (semi-transparent)
lc_rgba = np.concatenate([rgb, np.full((*rgb.shape[:2], 1), 180, dtype=np.uint8)], axis=-1)
ImageOverlay(
    image=lc_rgba,
    bounds=bounds,
    opacity=0.8,
    name='OEMJ land cover (RGB)',
    cross_origin=False,
).add_to(m_oemj)

# Tree mask layer (green with full alpha on trees)
mask_rgba = np.zeros((*tree_mask.shape, 4), dtype=np.uint8)
mask_rgba[..., 1] = 255  # green channel
mask_rgba[..., 3] = (tree_mask * 255).astype(np.uint8)
ImageOverlay(
    image=mask_rgba,
    bounds=bounds,
    opacity=0.9,
    name='Tree mask (OEMJ)',
    cross_origin=False,
).add_to(m_oemj)

folium.LayerControl().add_to(m_oemj)

m_oemj


In [ ]:
# Visualize canopy grid on Folium basemap
import folium
from folium.raster_layers import ImageOverlay
import matplotlib.cm as cm

# Normalize canopy for display (e.g., 0-20 m range)
display_vmin = 0.0
display_vmax = 20.0
valid = np.isfinite(canopy_refined)
clipped = np.clip(canopy_refined, display_vmin, display_vmax, where=valid, out=np.full_like(canopy_refined, np.nan))
norm = np.where(valid, (clipped - display_vmin) / max(display_vmax - display_vmin, 1e-6), np.nan)

# Colorize with viridis and make NaN transparent
viridis = cm.get_cmap('viridis')
rgba_float = viridis(np.nan_to_num(norm, nan=-1.0))
rgba = (rgba_float * 255).astype(np.uint8)
alpha = np.where(np.isfinite(norm), 255, 0).astype(np.uint8)
rgba[..., 3] = alpha

# Build bounds from AOI lon/lat used earlier
leaflet_bounds_canopy = [[lat_min, lon_min], [lat_max, lon_max]]
center_lat = (lat_min + lat_max) / 2.0
center_lon = (lon_min + lon_max) / 2.0

m_canopy = folium.Map(location=[center_lat, center_lon], zoom_start=15, tiles='CartoDB positron')
ImageOverlay(
    image=rgba,
    bounds=leaflet_bounds_canopy,
    opacity=0.85,
    name='Canopy height (0-20 m)',
    cross_origin=False,
).add_to(m_canopy)

# Add rectangle outline for AOI
folium.Rectangle(bounds=leaflet_bounds_canopy, color='#3388ff', weight=1, fill=False).add_to(m_canopy)
folium.LayerControl().add_to(m_canopy)

m_canopy
